# 01 — TİTCK ve SKRS Temizliği & Birleştirilmesi (İP1)

**Sorumlu:** Kişi A  
**Branch:** `feature/titck-skrs-temizlik-ve-birlestirme`  
**Amaç:** Türkiye'deki iki resmi ilaç listesini (TİTCK Ruhsatlı İlaçlar ve Sağlık Bakanlığı SKRS E-Reçete Listesi) temizleyip barkod üzerinden birleştirmek; mükerrerleri tekilleştirmek, ATC çelişkilerini analiz etmek ve ana `urun.csv` tablosunu üretmek.

---
### İP1 Kuralları Özeti:
- **Kural 1:** Ruhsatlı liste ana kaynaktır; SKRS durum, reçete türü ve ATC adı için kullanılır.
- **Kural 2:** Barkod temizliği yapılır, 13 hane ve EAN-13 kontrolü (`gtin_gecerli`) yapılır.
- **Kural 3:** TİTCK mükerrerlerinde askıda olmayan (0) ve tarihi en yeni olan tutulur, elenenler `barkod_tekrar.csv` dosyasına yazılır.
- **Kural 4:** ATC'si `V06` ve `V07` ile başlayanlar `kapsam_disi=1` işaretlenir.
- **Kural 5:** SKRS'te aktif olup ruhsatlıda olmayanlar eklenir (`kaynak='SKRS'`, `yol='SKRS'`).
- **Kural 6:** Boş kayıtlar SKRS ile tamamlanır, tamamlanamayanlar `eksik_kayit.csv` dosyasına yazılır.
- **Kural 8:** ATC çelişkilerinde hiyerarşik veya ruhsatlı kod `atc_final` olarak atanır.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# src modülünü bağla
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.normalize import tr_lower, barkod_temizle, gtin_gecerli

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('✓ Gerekli kütüphaneler ve src.normalize modülü başarıyla yüklendi.')

✓ Gerekli kütüphaneler ve src.normalize modülü başarıyla yüklendi.


## 1. Ham Verileri Yükleme
TİTCK Ruhsatlı listesi ve SKRS Aktif / Pasif sekmeleri okunur.

In [2]:
kok = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
r_yol = kok / 'data' / 'raw' / 'ruhsatli_2026-10-14.xlsx'
s_yol = kok / 'data' / 'raw' / 'skrs_2026-10-14.xlsx'

df_titck = pd.read_excel(r_yol, sheet_name='RUHSATLI ÜRÜNLER LİSTESİ', header=1, dtype=str)
df_skrs_aktif = pd.read_excel(s_yol, sheet_name='AKTİF ÜRÜNLER LİSTESİ', header=2, dtype=str)
df_skrs_pasif = pd.read_excel(s_yol, sheet_name='PASİF ÜRÜNLER LİSTESİ', header=2, dtype=str)

print(f'TİTCK Ruhsatlı Listesi: {df_titck.shape[0]:,d} satır, {df_titck.shape[1]} sütun')
print(f'SKRS Aktif Listesi:     {df_skrs_aktif.shape[0]:,d} satır, {df_skrs_aktif.shape[1]} sütun')
print(f'SKRS Pasif Listesi:     {df_skrs_pasif.shape[0]:,d} satır, {df_skrs_pasif.shape[1]} sütun')

TİTCK Ruhsatlı Listesi: 23,105 satır, 15 sütun
SKRS Aktif Listesi:     7,949 satır, 12 sütun
SKRS Pasif Listesi:     10,212 satır, 9 sütun


## 2. Barkod Temizliği ve TİTCK Tekrar Analizi (Kural 2 & 3)
Barkodlar standart sayısal stringe dönüştürülür. TİTCK mükerrer kayıtlarında askıda olmayan (`ASKIDA_DURUMU == '0'`) ve ruhsat tarihi en yeni olan tutulur, elenenler `barkod_tekrar.csv` dosyasına yazılır.

In [3]:
# Askı durum sütununu tespit et
aski_col = [c for c in df_titck.columns if 'ASKIDA' in str(c)][0]
df_titck['ASKIDA_DURUMU'] = df_titck[aski_col].fillna('0').str.strip()

# Barkod temizliği
df_titck['barkod'] = df_titck['BARKOD'].apply(barkod_temizle)
df_skrs_aktif['barkod'] = df_skrs_aktif['Barkod'].apply(barkod_temizle)
df_skrs_pasif['barkod'] = df_skrs_pasif['Barkod'].apply(barkod_temizle)

# TİTCK Mükerrer Yönetimi
df_titck['tarih_dt'] = pd.to_datetime(df_titck['RUHSAT TARİHİ'], errors='coerce')
df_titck['aski_prio'] = (df_titck['ASKIDA_DURUMU'] == '0').astype(int)

# Sırala: askıda olmayan önce (1), tarihi en yeni olan önce
titck_sirali = df_titck.sort_values(by=['aski_prio', 'tarih_dt'], ascending=[False, False])

mask_notna = titck_sirali['barkod'].notna()
titck_tekil_dolu = titck_sirali[mask_notna].drop_duplicates(subset=['barkod'], keep='first')
titck_tekrar = titck_sirali[mask_notna][titck_sirali[mask_notna].duplicated(subset=['barkod'], keep='first')].copy()
titck_bos = titck_sirali[~mask_notna]
titck_tekil = pd.concat([titck_tekil_dolu, titck_bos], ignore_index=True)

# Elenen mükerrerleri temiz formatta kaydet
tekrar_cols = ['BARKOD', 'ÜRÜN ADI', 'ETKİN MADDE', 'ATC KODU', 'RUHSAT SAHİBİ', 'RUHSAT TARİHİ', 'RUHSAT NUMARASI', 'ASKIDA_DURUMU']
titck_tekrar_clean = titck_tekrar[tekrar_cols].copy()
titck_tekrar_clean.rename(columns={
    'BARKOD': 'barkod', 'ÜRÜN ADI': 'urun_adi', 'ETKİN MADDE': 'etkin_madde',
    'ATC KODU': 'atc_kodu', 'RUHSAT SAHİBİ': 'ruhsat_sahibi', 'RUHSAT TARİHİ': 'ruhsat_tarihi',
    'RUHSAT NUMARASI': 'ruhsat_no', 'ASKIDA_DURUMU': 'askida_durumu'
}, inplace=True)
tekrar_yol = kok / 'data' / 'interim' / 'barkod_tekrar.csv'
titck_tekrar_clean.to_csv(tekrar_yol, index=False, encoding='utf-8')

print(f'TİTCK Başlangıç Satır Sayısı:        {len(df_titck):,d}')
print(f'Mükerrer Nedeniyle Elenen Satır:     {len(titck_tekrar_clean):,d} (-> data/interim/barkod_tekrar.csv)')
print(f'TİTCK Kalan Tekil Satır Sayısı:       {len(titck_tekil):,d}')

TİTCK Başlangıç Satır Sayısı:        23,105
Mükerrer Nedeniyle Elenen Satır:     52 (-> data/interim/barkod_tekrar.csv)
TİTCK Kalan Tekil Satır Sayısı:       23,053


## 3. SKRS Tekilleştirme
Aynı barkod hem Aktif hem Pasif sekmesinde yer alıyorsa, Kural gereği **Aktif** olan kayıt esas alınır.

In [4]:
df_skrs_aktif['durum'] = 'Aktif'
df_skrs_pasif['durum'] = 'Pasif'

skrs_hepsi = pd.concat([df_skrs_aktif, df_skrs_pasif], ignore_index=True)
skrs_hepsi['aktif_prio'] = (skrs_hepsi['durum'] == 'Aktif').astype(int)
skrs_sirali = skrs_hepsi.sort_values(by=['aktif_prio'], ascending=[False])
skrs_tekil = skrs_sirali.drop_duplicates(subset=['barkod'], keep='first').copy()

print(f'SKRS Aktif: {len(df_skrs_aktif):,d} | Pasif: {len(df_skrs_pasif):,d} | Tekil SKRS: {len(skrs_tekil):,d}')

SKRS Aktif: 7,949 | Pasif: 10,212 | Tekil SKRS: 18,161


## 4. Sol Birleştirme (Left Join), Kural 6 (Kurtarma) & Kural 5 (SKRS Ek Ürünler)
Ruhsatlı listeye SKRS barkod üzerinden soldan birleştirilir (`left join`). Kural 6 gereği etkin maddesi boş olan ruhsatlı ilaçlar SKRS ATC Adı ile tamamlanır. Ardından SKRS'te aktif olup ruhsatlıda olmayan kayıtlar tabloya eklenir.

In [5]:
skrs_sub = skrs_tekil[['barkod', 'ATC Kodu', 'ATC Adı', 'durum', 'İlaç Adı']].copy()
skrs_sub.rename(columns={
    'ATC Kodu': 'atc_skrs_raw',
    'ATC Adı': 'atc_adi_skrs',
    'durum': 'durum_skrs'
}, inplace=True)

merged = pd.merge(titck_tekil, skrs_sub, on='barkod', how='left')
merged['kaynak'] = 'Ruhsatlı'
merged['urun_adi'] = merged['ÜRÜN ADI']
merged['etkin_madde_ham'] = merged['ETKİN MADDE']
merged['atc_ruhsat_raw'] = merged['ATC KODU']
merged['durum'] = merged['durum_skrs'].where(merged['durum_skrs'].notna(), merged['ASKIDA_DURUMU'].map({'0': 'Aktif'}).fillna('Pasif'))

# Kural 6: Boş kayıt tamamlama (Etkin maddesi boş olan ruhsatlı ürün SKRS ATC Adı ile tamamlanır)
kural6_kurtarilan = merged['etkin_madde_ham'].isna() & merged['atc_adi_skrs'].notna()
if kural6_kurtarilan.sum() > 0:
    kurtarilan_ornek = merged.loc[kural6_kurtarilan, ['barkod', 'urun_adi', 'atc_adi_skrs']].iloc[0]
    print(f'Kural 6 Kurtarma: Barkod {kurtarilan_ornek["barkod"]} ({kurtarilan_ornek["urun_adi"]}) etkin maddesi SKRS ATC Adı ({kurtarilan_ornek["atc_adi_skrs"]}) ile tamamlandı.')
    merged.loc[kural6_kurtarilan, 'etkin_madde_ham'] = merged.loc[kural6_kurtarilan, 'atc_adi_skrs']

# Kural 5: SKRS'te aktif olup ruhsatlıda olmayanlar
titck_b_set = set(titck_tekil['barkod'].dropna())
skrs_eklenecek = skrs_tekil[(skrs_tekil['durum'] == 'Aktif') & (~skrs_tekil['barkod'].isin(titck_b_set))].copy()

ek_df = pd.DataFrame({
    'barkod': skrs_eklenecek['barkod'],
    'urun_adi': skrs_eklenecek['İlaç Adı'],
    'etkin_madde_ham': skrs_eklenecek['ATC Adı'],  # Kural 5 gereği İngilizce ATC adı
    'atc_ruhsat_raw': np.nan,
    'atc_skrs_raw': skrs_eklenecek['ATC Kodu'],
    'atc_adi_skrs': skrs_eklenecek['ATC Adı'],
    'durum': 'Aktif',
    'kaynak': 'SKRS'
})

urun_full = pd.concat([merged, ek_df], ignore_index=True)
print(f'Left Join Sonrası Ruhsatlı:          {len(merged):,d}')
print(f'Eklenen SKRS Aktif Ürün (Kural 5):   {len(ek_df):,d}')
print(f'Birleştirilmiş Toplam Ürün Sayısı:   {len(urun_full):,d}')

Kural 6 Kurtarma: Barkod 8699578092429 (LOXIBIN 50 MG FILM KAPLI TABLET, 28 ADET) etkin maddesi SKRS ATC Adı (losartan) ile tamamlandı.
Left Join Sonrası Ruhsatlı:          23,053
Eklenen SKRS Aktif Ürün (Kural 5):   851
Birleştirilmiş Toplam Ürün Sayısı:   23,904


## 5. ATC Temizliği, Çelişki Analizi & Kural 8
ATC kodlarındaki görünmez boşluklar temizlenir. TİTCK ve SKRS kodları 5 sınıfa ayrılarak çelişkiler tespit edilir ve nihai `atc_final` kodu üretilir.

In [6]:
def clean_atc(x):
    if pd.isna(x): return None
    s = tr_lower(x)
    return s.upper() if s else None

urun_full['atc_ruhsat'] = urun_full['atc_ruhsat_raw'].apply(clean_atc)
urun_full['atc_skrs'] = urun_full['atc_skrs_raw'].apply(clean_atc)

def celiski_sinifi(row):
    r = row['atc_ruhsat']
    s = row['atc_skrs']
    if pd.isna(r) or pd.isna(s): return 'bos'
    if r == s: return 'bosluk'
    if r.startswith(s) or s.startswith(r): return 'duzey_farki'
    if len(r) >= 5 and len(s) >= 5 and r[:5] == s[:5]: return 'ayni_grup_farkli_madde'
    return 'farkli_grup'

urun_full['atc_celiski_sinifi'] = urun_full.apply(celiski_sinifi, axis=1)

print('ATC Çelişki Sınıfları Dağılımı:')
print(urun_full['atc_celiski_sinifi'].value_counts())

# Kural 8: atc_final ve atc_celiski belirleme
def atc_final_belirle(row):
    r = row['atc_ruhsat']
    s = row['atc_skrs']
    sinif = row['atc_celiski_sinifi']
    if sinif == 'bosluk':
        return r, 0
    elif sinif == 'duzey_farki':
        return (r if len(r) >= len(s) else s), 0
    elif sinif in ('ayni_grup_farkli_madde', 'farkli_grup'):
        return r, 1
    elif sinif == 'bos':
        return (r if pd.notna(r) else s), 0
    return r, 0

finals = urun_full.apply(atc_final_belirle, axis=1)
urun_full['atc_final'] = [f[0] for f in finals]
urun_full['atc_celiski'] = [f[1] for f in finals]

# Çelişki tablosunu kaydet
celiski_df = urun_full[urun_full['atc_celiski_sinifi'] != 'bosluk'][[
    'barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs', 'atc_adi_skrs', 'atc_celiski_sinifi'
]].copy()
celiski_df.to_csv(kok / 'data' / 'interim' / 'atc_celiski_siniflari.csv', index=False, encoding='utf-8')
print(f'atc_celiski_siniflari.csv kaydedildi ({len(celiski_df):,d} satır).')

ATC Çelişki Sınıfları Dağılımı:
atc_celiski_sinifi
bos                       12264
bosluk                    11222
farkli_grup                 342
duzey_farki                  49
ayni_grup_farkli_madde       27
Name: count, dtype: int64


atc_celiski_siniflari.csv kaydedildi (12,682 satır).


## 6. Kural 4 (Kapsam Dışı) & Kural 6 (Eksik Kayıtlar)
- ATC kodu `V06` veya `V07` ile başlayanlar `kapsam_disi = 1` işaretlenir.
- Etkin madde veya ATC kodu boş olan ve tamamlanamayanlar `eksik_kayit.csv` dosyasına yazılır.

In [7]:
# Kural 4: V06 / V07 kapsam dışı
urun_full['kapsam_disi'] = urun_full['atc_final'].fillna('').str.startswith(('V06', 'V07')).astype(int)

# Kural 2: GTIN Kontrol hanesi doğrulaması
urun_full['barkod_gecersiz'] = (~urun_full['barkod'].apply(gtin_gecerli)).astype(int)

# Kural 6: Tamamlanamayan eksik kayıtlar
c_etkin = urun_full['etkin_madde_ham'].isna()
c_atc = urun_full['atc_final'].isna()
eksik_df = urun_full[c_etkin | c_atc].copy()

def eksik_turu(row):
    e = pd.isna(row['etkin_madde_ham'])
    a = pd.isna(row['atc_final'])
    if e and a: return 'etkin_madde_ve_atc_yok'
    if e: return 'etkin_madde_yok'
    return 'atc_yok'

eksik_df['eksik_turu'] = eksik_df.apply(eksik_turu, axis=1)
eksik_cols = ['barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs', 'atc_adi_skrs', 'durum', 'kaynak', 'eksik_turu']
eksik_clean = eksik_df[eksik_cols].copy()
eksik_clean.to_csv(kok / 'data' / 'interim' / 'eksik_kayit.csv', index=False, encoding='utf-8')

print(f'Kapsam Dışı (V06/V07) Ürün Sayısı:  {urun_full["kapsam_disi"].sum():,d}')
print(f'Eksik Kayıt Sayısı:                 {len(eksik_clean):,d} (-> data/interim/eksik_kayit.csv)')
print(f'Geçersiz GTIN Barkod Sayısı:        {urun_full["barkod_gecersiz"].sum():,d}')

Kapsam Dışı (V06/V07) Ürün Sayısı:  456
Eksik Kayıt Sayısı:                 103 (-> data/interim/eksik_kayit.csv)
Geçersiz GTIN Barkod Sayısı:        135


## 7. Nihai `urun.csv` Tablosunu Kaydetme
Standart sütun formatıyla `data/interim/urun.csv` dosyası üretilir.

In [8]:
final_cols = [
    'barkod', 'urun_adi', 'etkin_madde_ham', 'atc_ruhsat', 'atc_skrs',
    'atc_adi_skrs', 'durum', 'kaynak', 'atc_final', 'atc_celiski',
    'kapsam_disi', 'barkod_gecersiz'
]
urun_df = urun_full[final_cols].copy()
cikis_yol = kok / 'data' / 'interim' / 'urun.csv'
urun_df.to_csv(cikis_yol, index=False, encoding='utf-8')

print(f'✓ {cikis_yol} başarıyla kaydedildi!')
print(f'Toplam Satır Sayısı: {len(urun_df):,d}')
display(urun_df.head(5))

✓ /Users/mervesudeyanar/Desktop/ddi-final/data/interim/urun.csv başarıyla kaydedildi!
Toplam Satır Sayısı: 23,904


,barkod,urun_adi,etkin_madde_ham,atc_ruhsat,atc_skrs,atc_adi_skrs,durum,kaynak,atc_final,atc_celiski,kapsam_disi,barkod_gecersiz
0,8685276183408,"GLİMATİNE 400 MG FİLM KAPLI TABLET, 30 ADET",İmatinib mesilat,L01EA01,None,NaN,Aktif,Ruhsatlı,L01EA01,0,0,0
1,8681990009376,KSİLİDİN %5 MERHEM,lidocaine,D04AB01,None,NaN,Aktif,Ruhsatlı,D04AB01,0,0,0
2,8685119899022,ADELEKS 4 MG/2 ML IM ENJEKSİYONLUK ÇÖZELTİ İÇE...,tiyokolşikosid,M03BX05,None,NaN,Aktif,Ruhsatlı,M03BX05,0,0,0
3,8699514640004,"TANFLEX PLUS % 0.15+% 0.12 GARGARA, 120 ML","benzidamin hidroklorür, klorheksidin glukonat",A01AD11,None,NaN,Aktif,Ruhsatlı,A01AD11,0,0,0
4,8699514640097,"TANFLEX PLUS % 0.15+% 0.12 GARGARA, 200 ML","benzidamin hidroklorür, klorheksidin glukonat",A01AD11,None,NaN,Aktif,Ruhsatlı,A01AD11,0,0,0


## 8. İP1 Bitti Sayılır — Özet Rapor Tablosu
PDF gereksinimindeki kontrol tablosu oluşturulur.

In [9]:
ozet_tablo = pd.DataFrame([
    {'Metrik': 'TİTCK Başlangıç Satır Sayısı', 'Değer': len(df_titck)},
    {'Metrik': 'Mükerrer Barkod Nedeniyle Elenen (TİTCK)', 'Değer': len(titck_tekrar_clean)},
    {'Metrik': 'TİTCK Tekil Ürün Sayısı', 'Değer': len(titck_tekil)},
    {'Metrik': "SKRS'ten Eklenen Aktif Ürünler (Kural 5)", 'Değer': len(ek_df)},
    {'Metrik': 'Kural 6 ile Kurtarılan Kayıt (LOXIBIN)', 'Değer': int(kural6_kurtarilan.sum())},
    {'Metrik': 'Kapsam Dışı İşaretlenen (V06/V07)', 'Değer': int(urun_full['kapsam_disi'].sum())},
    {'Metrik': 'Eksik Kayıt Sayısı (Kurtarılamayan)', 'Değer': len(eksik_clean)},
    {'Metrik': 'Nihai Ürün Sayısı (urun.csv)', 'Değer': len(urun_df)},
    {'Metrik': 'ATC Çelişkisi: bosluk (birebir aynı)', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'bosluk').sum())},
    {'Metrik': 'ATC Çelişkisi: duzey_farki', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'duzey_farki').sum())},
    {'Metrik': 'ATC Çelişkisi: ayni_grup_farkli_madde', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'ayni_grup_farkli_madde').sum())},
    {'Metrik': 'ATC Çelişkisi: farkli_grup', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'farkli_grup').sum())},
    {'Metrik': 'ATC Çelişkisi: bos (bir tarafta yok)', 'Değer': int((urun_full['atc_celiski_sinifi'] == 'bos').sum())},
])
display(ozet_tablo)

,Metrik,Değer
0,TİTCK Başlangıç Satır Sayısı,23105
1,Mükerrer Barkod Nedeniyle Elenen (TİTCK),52
2,TİTCK Tekil Ürün Sayısı,23053
3,SKRS'ten Eklenen Aktif Ürünler (Kural 5),851
4,Kural 6 ile Kurtarılan Kayıt (LOXIBIN),1
5,Kapsam Dışı İşaretlenen (V06/V07),456
6,Eksik Kayıt Sayısı (Kurtarılamayan),103
7,Nihai Ürün Sayısı (urun.csv),23904
8,ATC Çelişkisi: bosluk (birebir aynı),11222
9,ATC Çelişkisi: duzey_farki,49
